# Proyecto 3: Mejorando algoritmos con CUDA

CC3069 Computación Paralela y Distribuida. Integrantes: Diego Lopez, Diego Rosales, Mia Fuentes.

Este notebook **no entrena nada**: lee los resultados que cada fase dejó en `resultados/` y arma la comparación final.

Flujo del pipeline:
1. **Preparación de datos (CPU)**: `comun/datos_seq.py` (secuencial) vs `fase_a_preparacion_cpu/` (multinúcleo).
2. **Configuración CUDA y entrenamiento (GPU)**: `fase_b_entrenamiento_gpu/`.
3. **Kernels CUDA propios (GPU)**: `fase_c_kernels_numba/`.

## 1. Análisis de la solución secuencial (Mia)

Ver `analisis/operaciones.md` y `analisis/oportunidades.md`.

In [ ]:
import glob
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from comun import config
from comun.medicion import validar_variante

df = pd.concat([pd.read_csv(p).assign(archivo=p) for p in glob.glob('resultados/tiempos_*.csv')], ignore_index=True)
for fase, variante in df[['fase', 'variante']].drop_duplicates().itertuples(index=False):
    validar_variante(fase, variante)  # falla si alguien usó un nombre fuera del contrato
df['hardware'].value_counts()

### Revisión de consistencia

Antes de comparar se verifica que cada variante tenga `N_CORRIDAS` corridas y que dentro de una fase y una máquina no se mezclen `batch`, `epocas` o `n_imagenes` distintos.

In [ ]:
conteo = df.groupby(['hardware', 'fase', 'variante']).size()
incompletas = conteo[conteo != config.N_CORRIDAS]
if not incompletas.empty:
    warnings.warn(f'Variantes con un número de corridas distinto de {config.N_CORRIDAS}:\n{incompletas}')

PARAMS = {'B_entrenamiento': ['batch', 'epocas', 'n_imagenes'], 'T_original': ['batch', 'epocas', 'n_imagenes'],
          'A_preparacion': ['n_imagenes'], 'C_inferencia': ['n_imagenes']}
for fase, cols in PARAMS.items():
    sub = df[df.fase == fase]
    for col in cols:
        distintos = sub.groupby('hardware')[col].nunique()
        if (distintos > 1).any():
            warnings.warn(f'{fase}: valores distintos de {col} en la misma máquina: {distintos[distintos > 1].to_dict()}')
incompletas

## 2. Tabla comparativa de tiempos

Siempre agrupada por máquina (`hardware`): un tiempo solo se compara con otro medido en el mismo equipo.

In [ ]:
tabla = (df.groupby(['hardware', 'fase', 'variante'])['tiempo_s']
           .agg(media_s='mean', desv_s='std', corridas='count')
           .reset_index())
tabla

## 3. Speedup por fase

S = T_base / T_variante, con la base de `config.BASE_SPEEDUP`, calculado **dentro de cada máquina**. Si una máquina no midió la variante base de una fase, esa fase no tiene speedup en esa máquina (se avisa).

In [ ]:
filas = []
for (hw, fase), g in tabla.groupby(['hardware', 'fase']):
    base = config.BASE_SPEEDUP.get(fase)
    if base is None:
        continue
    t = g.set_index('variante')['media_s']
    if base not in t:
        warnings.warn(f'{hw}: la fase {fase} no tiene la variante base {base!r}; se omite su speedup.')
        continue
    for var, val in t.items():
        filas.append({'hardware': hw, 'fase': fase, 'variante': var, 'media_s': val, 'speedup': t[base] / val})
speedups = pd.DataFrame(filas)
speedups

### Speedup total (extremo a extremo)

- **Antes**: `T_original / cpu_1hilo`, el pipeline original tal cual (lee los JPG en cada época, 1 hilo).
- **Después**: mejor `A_preparacion / procesos_<p>` + `B_entrenamiento / cuda` (o `cuda_amp` si es más rápido).

Solo se calcula en máquinas que midieron las tres cosas; la tabla oficial usa la laptop de Rosales.

In [ ]:
def speedup_total(t):
    m = t.set_index(['fase', 'variante'])['media_s']
    antes = m.get(('T_original', 'cpu_1hilo'))
    prep = m.loc['A_preparacion'].filter(like='procesos_').min() if 'A_preparacion' in m.index.get_level_values(0) else None
    entr = m.loc['B_entrenamiento'].reindex(['cuda', 'cuda_amp']).min() if 'B_entrenamiento' in m.index.get_level_values(0) else None
    if antes is None or pd.isna(prep) or pd.isna(entr) or prep is None or entr is None:
        return None
    return {'antes_s': antes, 'prep_paralela_s': prep, 'entrenamiento_gpu_s': entr,
            'despues_s': prep + entr, 'speedup_total': antes / (prep + entr)}

total = pd.DataFrame({hw: r for hw, g in tabla.groupby('hardware') if (r := speedup_total(g))}).T
total

In [ ]:
for (hw, fase), g in speedups.groupby(['hardware', 'fase']):
    ax = g.plot.bar(x='variante', y='speedup', legend=False, title=f'Speedup {fase}\n{hw}')
    ax.axhline(1, color='gray', lw=0.8)
    ax.set_ylabel('veces más rápido')
    plt.tight_layout(); plt.savefig(f'resultados/figuras/speedup_{fase}.png', dpi=150); plt.show()

## 4. Evidencia de uso de recursos

Figuras generadas con `comun.monitor.graficar` y capturas en `resultados/capturas/`.

In [ ]:
from comun.monitor import graficar
for log in sorted(glob.glob('resultados/logs/*.csv'))[:6]:
    print(graficar(log))

## 5. Discusión técnica

- P1 (Rosales): ¿por qué procesar imágenes con los núcleos del CPU y delegar la matemática de la red a la GPU?
- P2 (Lopez): ¿cómo CUDA permitió reducir o cambiar el tiempo de procesamiento?
- P3 (Mia): ¿qué factores justifican o limitan la utilización de CUDA?

## 6. Conclusiones